# 02 — Baseline Models (Mean/Majority, Linear/Logistic, Random Forest)

Uses deterministic 70/15/15 split with train-only preprocessing to prevent leakage.


In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from jdcl.models.baselines import (
    deterministic_split, TabularPreprocessor, MeanMajorityBaseline,
    RidgeLinearBaseline, LogisticBaseline, RandomForestBaseline,
    regression_metrics, classification_and_calibration_metrics,
)
df = pd.read_csv('../data/synthetic/jflm_2027.csv')
split = deterministic_split(df, seed=20270101)
prep = TabularPreprocessor().fit(split.train_df)
X_tr, X_val, X_te = prep.transform(split.train_df), prep.transform(split.val_df), prep.transform(split.test_df)
y_tr, y_te = split.train_df['resolution_days'].to_numpy(float), split.test_df['resolution_days'].to_numpy(float)
yc_tr, yc_te = split.train_df['service_completed'].to_numpy(int), split.test_df['service_completed'].to_numpy(int)
m_mean = MeanMajorityBaseline().fit(y_tr, yc_tr)
m_ridge = RidgeLinearBaseline(alpha=1.0).fit(X_tr, y_tr)
m_rf = RandomForestBaseline(n_estimators=20, max_depth=6, seed=20270101).fit(X_tr, y_tr)
m_log = LogisticBaseline().fit(X_tr, yc_tr)
print('Test Regression — Mean:', regression_metrics(y_te, m_mean.predict(X_te)))
print('Test Regression — Ridge:', regression_metrics(y_te, m_ridge.predict(X_te)))
print('Test Regression — RandomForest:', regression_metrics(y_te, m_rf.predict(X_te)))
print('Test Classification/Calibration — Logistic:', classification_and_calibration_metrics(yc_te, m_log.predict_proba(X_te)))


Test Regression — Mean: {'mae': 18.6559, 'rmse': 24.0615, 'r2': -0.0}
Test Regression — Ridge: {'mae': 7.1718, 'rmse': 9.1008, 'r2': 0.8569}
Test Regression — RandomForest: {'mae': 9.5241, 'rmse': 11.9581, 'r2': 0.753}
Test Classification/Calibration — Logistic: {'accuracy': 0.9147, 'f1': 0.955, 'brier_score': 0.0653, 'ece': 0.017}
